In [0]:
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")

In [0]:
inputCatalog = dbutils.widgets.get("catalog")
inpuSchema = dbutils.widgets.get("schema")

In [0]:
%sql
USE $catalog.$schema

In [0]:
%sql
CREATE OR REPLACE FUNCTION classify_store_performance()
RETURNS TABLE(
  location_id STRING,
  location_name STRING,
  forecast_date DATE,
  actual_revenue DOUBLE,
  forecast_revenue DOUBLE,
  variance_amount DOUBLE,
  variance_pct DOUBLE,
  performance_tier STRING,
  action_priority INT
)
COMMENT 'Classifies all stores into performance tiers based on forecast variance with prioritization'
RETURN 
  SELECT 
    isa.location_id,
    sl.location_name,
    isa.forecast_date,
    isa.total_sales AS actual_revenue,
    isa.forecast_sales_value AS forecast_revenue,
    isa.total_sales - isa.forecast_sales_value AS variance_amount,
    CASE 
      WHEN isa.forecast_sales_value = 0 OR isa.forecast_sales_value IS NULL THEN NULL
      ELSE ((isa.total_sales - isa.forecast_sales_value) / isa.forecast_sales_value) * 100
    END AS variance_pct,
    CASE 
      WHEN isa.total_sales IS NULL OR isa.forecast_sales_value IS NULL OR isa.forecast_sales_value = 0 
        THEN 'Insufficient Data'
      WHEN ((isa.total_sales - isa.forecast_sales_value) / isa.forecast_sales_value) > 0.15 
        THEN 'Outperformer'
      WHEN ((isa.total_sales - isa.forecast_sales_value) / isa.forecast_sales_value) > 0.05 
        THEN 'Above Target'
      WHEN ABS((isa.total_sales - isa.forecast_sales_value) / isa.forecast_sales_value) <= 0.05 
        THEN 'On Target'
      WHEN ((isa.forecast_sales_value - isa.total_sales) / isa.forecast_sales_value) > 0.05 
        AND ((isa.forecast_sales_value - isa.total_sales) / isa.forecast_sales_value) <= 0.15 
        THEN 'Below Target'
      WHEN ((isa.forecast_sales_value - isa.total_sales) / isa.forecast_sales_value) > 0.15 
        THEN 'At Risk'
      ELSE 'Unclassified'
    END AS performance_tier,
    CASE 
      WHEN isa.total_sales IS NULL OR isa.forecast_sales_value IS NULL OR isa.forecast_sales_value = 0 
        THEN 5
      WHEN ((isa.forecast_sales_value - isa.total_sales) / isa.forecast_sales_value) > 0.15 
        THEN 1  -- Critical: At Risk
      WHEN ((isa.forecast_sales_value - isa.total_sales) / isa.forecast_sales_value) > 0.05 
        THEN 2  -- High: Below Target
      WHEN ABS((isa.total_sales - isa.forecast_sales_value) / isa.forecast_sales_value) <= 0.05 
        THEN 3  -- Medium: On Target
      WHEN ((isa.total_sales - isa.forecast_sales_value) / isa.forecast_sales_value) > 0.05 
        THEN 4  -- Low: Above Target/Outperformer
      ELSE 5
    END AS action_priority
  FROM items_sales_aggregated_stores isa
  JOIN store_location sl ON isa.location_id = sl.location_id
  WHERE isa.total_sales IS NOT NULL 
    AND isa.forecast_sales_value IS NOT NULL;